# Évaluation WESAD sur Google Colab — sans rien télécharger chez soi

WESAD pèse **2,5 Go**. Ce notebook le télécharge **sur les serveurs de Google** (connexion très rapide), calcule la VFC (RMSSD) par fenêtre de 60 s pour les 15 participants, puis ne renvoie que deux petits fichiers :

- `wesad_hrv_windows.csv` (quelques centaines de Ko) : à copier dans `backend/data/physionet/` ; le seed l'importe en base ;
- `calibration.json` : sensibilité et taux de fausses alertes de la règle `HRV-01`.

**Mode d'emploi** : ouvrir ce fichier dans https://colab.research.google.com (Fichier → Importer le notebook), puis *Exécution → Tout exécuter*. Durée : environ 10 à 15 minutes.

**Conditions d'utilisation de WESAD** : usage scientifique et non commercial, en citant Schmidt et al., *Introducing WESAD, a Multimodal Dataset for Wearable Stress and Affect Detection*, ICMI 2018.

## 1. Récupérer le code d'Astro Assistant

In [ ]:
!git clone -q https://github.com/alexis-fokam/astroassistant-backend.git
%cd astroassistant-backend
!pip install -q -e .

## 2. Télécharger et décompresser WESAD (côté Google)

In [1]:
# Lien officiel publié par l'université de Siegen (partage Sciebo).
!wget -c -q --show-progress -O /content/WESAD.zip "https://uni-siegen.sciebo.de/s/HGdUkoNlW1Ub0Gx/download"
!mkdir -p data/external && unzip -q -o /content/WESAD.zip -d /content/wesad_raw
# Les dossiers S2 … S17 doivent se trouver directement dans data/external/wesad/.
import pathlib
import shutil

root = next(p.parent.parent for p in pathlib.Path("/content/wesad_raw").rglob("S2.pkl"))
target = pathlib.Path("data/external/wesad")
target.mkdir(parents=True, exist_ok=True)
for subject in sorted(root.glob("S*")):
    if subject.is_dir() and not (target / subject.name).exists():
        shutil.move(str(subject), target / subject.name)
print(sorted(p.name for p in target.iterdir()))

'wget' is not recognized as an internal or external command,
operable program or batch file.
The syntax of the command is incorrect.


StopIteration: 

## 3. Extraire la VFC par fenêtre de 60 s, puis évaluer la règle HRV-01

In [ ]:
!python -m app.engine.calibration wesad-extract
!python -m app.engine.calibration wesad

In [ ]:
import json

report = json.load(open("docs/calibration.json", encoding="utf-8"))["wesad"]
print("Participants :", report["subjects"])
print("RMSSD stress / repos (médiane) :", report["median_rmssd_stress_over_baseline"])
for r in report["hrv_rule"]:
    print(
        f"seuil {r['ratio']:.0%} de la baseline → sensibilité {r['sensitivity']:.0%}, fausses alertes {r['false_alarm_rate']:.0%}"
    )

## 4. Récupérer les deux petits fichiers

In [ ]:
from google.colab import files

files.download("data/physionet/wesad_hrv_windows.csv")
files.download("docs/calibration.json")

Ensuite, sur ton ordinateur :

1. copie `wesad_hrv_windows.csv` dans `backend/data/physionet/` et `calibration.json` dans `backend/docs/` ;
2. lance `python -m app.seed --reset` : les fenêtres WESAD entrent dans la table `reference_vital_samples` ;
3. commite les deux fichiers.